# 3D weld pool: heat, flow, solidification

Half-plate model for aluminium arc welding of a **monolithic plate** (joint gap not resolved).

**This model:**
- 3D half-domain: \(x\) along weld, \(y\) from symmetry plane, \(z\) thickness
- Arc replaced by a **moving Gaussian surface heat flux** with full-plane power \(P\)
- Liquid / solid via linear liquid fraction + Darcy sink in the mushy zone
- Driving forces: **Marangoni** (\(u\) and \(v\)) + buoyancy (Boussinesq)
- **No** electromagnetism, arc pressure, free-surface deformation, or filler addition

**References** (`refs/weld_pool`):
- He, Fuerschbach, DebRoy, *J. Phys. D* 36 (2003)
- Zhao & DebRoy, *Metall. Mater. Trans. B* 32B (2001) — Al 5182, negative \(d\sigma/dT\)
- Pitscheneder et al., *Welding Journal* (1996)
- Kidess et al., *Int. J. Thermal Sci.* 104 (2016)


## Схема постановки

Половина пластины; плоскость симметрии \(y=0\) — ось шва.

```
        q(x,y,t)  Gaussian, moves with arc along +x
              |||||  →  v_weld
    z=Lz  ─────████████████████──────────── top (Marangoni)
              │  liquid / mush / solid   │
    z↑        │                          │
    z=0   ────┴──────────────────────────┴──── bottom
         x=0 ──────────→ x ──────────────── Lx

  y=0 symmetry (v=0)  ···  y = Ly (wall)
  Arc: x_c(t)=x_0+v t, y_c=0
  q_max = 3 P / (π r_b²)  →  ∫q dA = P over full plane
```

| Символ | Смысл | Типично |
|---|---|---|
| \(x\) | вдоль шва | 0…20 mm |
| \(y\) | от оси шва | 0…8 mm |
| \(z\) | толщина | 0…4 mm |
| \(P\) | мощность дуги | ~2 kW |
| \(r_b, v\) | радиус пятна, скорость | мм, мм/с |


## Equations

Continuity and momentum (He et al.), three velocity components:

$$
\rho \partial_t \mathbf{V}
= -\rho \nabla\cdot(\mathbf{V}\mathbf{V})
+ \nabla\cdot(\mu\nabla\mathbf{V})
- \nabla p
- C\frac{(1-f_L)^2}{f_L^3+B}\mathbf{V}
+ \rho g\beta(T-T_\mathrm{ref})\,\mathbf{e}_z
$$

Energy with apparent heat capacity in the mush:

$$
\rho c_{p,\mathrm{eff}}\partial_t T
+ \rho c_p \mathbf{V}\cdot\nabla T
= \nabla\cdot(k\nabla T),
\qquad
c_{p,\mathrm{eff}} = c_p + \frac{L}{T_l-T_s}\ \text{(in mush)}
$$

Top surface:

$$
q = q_\max\exp\Big(-3\frac{(x-x_c)^2+y^2}{r_b^2}\Big)
- h(T-T_a) - \varepsilon\sigma(T^4-T_a^4)
$$

$$
\mu\partial_z u = f_L\,\frac{d\sigma}{dT}\,\partial_x T,
\quad
\mu\partial_z v = f_L\,\frac{d\sigma}{dT}\,\partial_y T,
\quad w=0
$$

Symmetry at \(y=0\): \(v=0\), insulated. Sides / ends: no-slip, insulated. Bottom: convection.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if ROOT.name != 'heat+flow_3dmodel':
    cand = ROOT / 'heat+flow_3dmodel'
    if cand.is_dir():
        sys.path.insert(0, str(cand))
        MODEL_DIR = cand
    else:
        sys.path.insert(0, str(ROOT))
        MODEL_DIR = ROOT
else:
    sys.path.insert(0, str(ROOT))
    MODEL_DIR = ROOT

import importlib
import weldpool3d

importlib.reload(weldpool3d)
from weldpool3d import Material, Process, WeldPool3D, quick_process

print('model dir:', MODEL_DIR)
print('Diagnostics fields include T_frames:', hasattr(weldpool3d.Diagnostics, '__dataclass_fields__') and 'T_frames' in weldpool3d.Diagnostics.__dataclass_fields__)


## Material (Al 5182) and process parameters

Defaults from Zhao & DebRoy (2001).  
`QUICK = True` — coarse short case. Set `False` for a finer default grid (still a demo, not a full arc passport).


In [ ]:
QUICK = False

mat = Material()

if QUICK:
    proc = quick_process(
        enable_flow=True,
        nx=24,
        ny=10,
        nz=8,
        t_end=0.35,
        P_arc=2200.0,
        r_b=0.0025,
        v_weld=0.008,
        x0=0.005,
    )
else:
    proc = Process(enable_flow=True, t_end=1.0)

model = WeldPool3D(material=mat, process=proc)
print(
    f'grid {proc.nx}×{proc.ny}×{proc.nz}, '
    f'Domain {1e3*proc.Lx:.0f}×{1e3*proc.Ly:.0f}×{1e3*proc.Lz:.0f} mm, '
    f'P={proc.P_arc:.0f} W, q_max={proc.q_max:.2e} W/m²'
)


## Run


In [ ]:
diag = model.run(record_every=0.05, progress=True, record_fields=True)
depth, length, width = model.pool_metrics()
print(
    f'final: Tmax={model.T.max():.0f} K, depth={1e3*depth:.2f} mm, '
    f'length={1e3*length:.2f} mm, width={1e3*width:.2f} mm, '
    f'heat bal={model.heat_balance_ratio():.3f}'
)
print(f'frames recorded: {len(diag.T_frames)}')


## Fields: mid-plane \(y=0\) and cross-section at arc


In [ ]:
u_c, v_c, w_c = model._cell_velocity()
speed = np.sqrt(u_c**2 + v_c**2 + w_c**2)

j0 = 0  # near symmetry
Xc = 1e3 * model.X[:, j0, :]
Zc = 1e3 * model.Z[:, j0, :]

ix_arc = int(np.clip(np.searchsorted(model.xc, model.arc_centre()) - 1, 0, proc.nx - 1))
Yc = 1e3 * model.Y[ix_arc, :, :]
Zc_cs = 1e3 * model.Z[ix_arc, :, :]

fig, axs = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True)

im = axs[0, 0].pcolormesh(Xc, Zc, model.T[:, j0, :], shading='auto', cmap='inferno')
axs[0, 0].contour(Xc, Zc, model.T[:, j0, :], levels=[mat.T_solidus, mat.T_liquidus], colors=['c', 'lime'])
axs[0, 0].axvline(1e3 * model.arc_centre(), color='w', ls='--', lw=0.8)
fig.colorbar(im, ax=axs[0, 0], label='T [K]')
axs[0, 0].set_title(f'T mid-plane y≈0, t={model.t:.2f}s')

im = axs[0, 1].pcolormesh(Xc, Zc, model.fL[:, j0, :], shading='auto', cmap='Blues', vmin=0, vmax=1)
fig.colorbar(im, ax=axs[0, 1], label=r'$f_L$')
axs[0, 1].set_title(r'$f_L$ mid-plane')

im = axs[0, 2].pcolormesh(Xc, Zc, speed[:, j0, :], shading='auto', cmap='viridis')
fig.colorbar(im, ax=axs[0, 2], label='|V| [m/s]')
axs[0, 2].set_title('|V| mid-plane')

im = axs[1, 0].pcolormesh(Yc, Zc_cs, model.T[ix_arc, :, :], shading='auto', cmap='inferno')
axs[1, 0].contour(Yc, Zc_cs, model.T[ix_arc, :, :], levels=[mat.T_solidus, mat.T_liquidus], colors=['c', 'lime'])
fig.colorbar(im, ax=axs[1, 0], label='T [K]')
axs[1, 0].set_title(f'T cross-section x≈x_c')

im = axs[1, 1].pcolormesh(Yc, Zc_cs, model.fL[ix_arc, :, :], shading='auto', cmap='Blues', vmin=0, vmax=1)
fig.colorbar(im, ax=axs[1, 1], label=r'$f_L$')
axs[1, 1].set_title(r'$f_L$ cross-section')

# top-view T
Xt, Yt = 1e3 * model.X[:, :, -1], 1e3 * model.Y[:, :, -1]
im = axs[1, 2].pcolormesh(Xt, Yt, model.T[:, :, -1], shading='auto', cmap='inferno')
axs[1, 2].contour(Xt, Yt, model.T[:, :, -1], levels=[mat.T_solidus, mat.T_liquidus], colors=['c', 'lime'])
fig.colorbar(im, ax=axs[1, 2], label='T [K]')
axs[1, 2].set_title('T top surface')

for ax in axs[0, :]:
    ax.set_xlabel('x [mm]'); ax.set_ylabel('z [mm]'); ax.set_aspect('equal')
axs[1, 0].set_xlabel('y [mm]'); axs[1, 0].set_ylabel('z [mm]'); axs[1, 0].set_aspect('equal')
axs[1, 1].set_xlabel('y [mm]'); axs[1, 1].set_ylabel('z [mm]'); axs[1, 1].set_aspect('equal')
axs[1, 2].set_xlabel('x [mm]'); axs[1, 2].set_ylabel('y [mm]'); axs[1, 2].set_aspect('equal')
plt.show()


## Pool size and thermal cycle


In [ ]:
t = np.array(diag.time)
fig, axs = plt.subplots(1, 3, figsize=(11, 3.2), constrained_layout=True)
axs[0].plot(t, 1e3 * np.array(diag.pool_depth), label='depth')
axs[0].plot(t, 1e3 * np.array(diag.pool_length), label='length')
axs[0].plot(t, 1e3 * np.array(diag.pool_width), label='width (full)')
axs[0].set_xlabel('t [s]'); axs[0].set_ylabel('mm'); axs[0].legend(fontsize=8)
axs[0].set_title('Pool size'); axs[0].grid(True, alpha=0.3)

axs[1].plot(t, diag.T_probe, color='C1')
axs[1].axhline(mat.T_solidus, color='c', ls='--', label=r'$T_s$')
axs[1].axhline(mat.T_liquidus, color='g', ls='--', label=r'$T_l$')
axs[1].set_xlabel('t [s]'); axs[1].set_ylabel('T [K]'); axs[1].legend(fontsize=8)
axs[1].set_title('Probe thermal cycle'); axs[1].grid(True, alpha=0.3)

axs[2].plot(t, diag.max_speed, label='|V|max')
axs[2].plot(t, diag.pe_number, label='Pe')
axs[2].set_xlabel('t [s]'); axs[2].legend(fontsize=8)
axs[2].set_title('Flow intensity'); axs[2].grid(True, alpha=0.3)
plt.show()

print(f'v=0 on symmetry: max |v(y=0)| = {np.max(np.abs(model.v[:, 0, :])):.2e} m/s')
liq = model.fL > 0.5
div = model.divergence()
print(f'max |div| in liquid: {np.max(np.abs(div[liq])) if np.any(liq) else 0:.2e} 1/s')


## Animation over time

Requires `record_fields=True` in the run cell. Mid-plane \(y\approx 0\), cross-section at the moving arc, and top surface. Use the player controls below the figure.


In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

assert diag.T_frames, 'No field frames — re-run with record_fields=True'

times = np.array(diag.time)
T_fr = diag.T_frames
fL_fr = diag.fL_frames
spd_fr = diag.speed_frames
arc_fr = np.array(diag.arc_x_frames)

j0 = 0
Xc = 1e3 * model.X[:, j0, :]
Zc = 1e3 * model.Z[:, j0, :]
Xt = 1e3 * model.X[:, :, -1]
Yt = 1e3 * model.Y[:, :, -1]

T_vmin = float(min(f.min() for f in T_fr))
T_vmax = float(max(f.max() for f in T_fr))
spd_vmax = float(max(f.max() for f in spd_fr)) or 1e-3

fig, axs = plt.subplots(1, 3, figsize=(12, 3.4), constrained_layout=True)

im0 = axs[0].pcolormesh(Xc, Zc, T_fr[0][:, j0, :], shading='auto', cmap='inferno', vmin=T_vmin, vmax=T_vmax)
ct0 = axs[0].contour(Xc, Zc, T_fr[0][:, j0, :], levels=[mat.T_solidus, mat.T_liquidus], colors=['c', 'lime'])
arc_line = axs[0].axvline(1e3 * arc_fr[0], color='w', ls='--', lw=0.8)
fig.colorbar(im0, ax=axs[0], label='T [K]')
axs[0].set_xlabel('x [mm]'); axs[0].set_ylabel('z [mm]'); axs[0].set_aspect('equal')
title0 = axs[0].set_title(f'T mid-plane, t={times[0]:.2f}s')

# cross-section y–z at arc x (index updated each frame)
ix0 = int(np.clip(np.searchsorted(model.xc, arc_fr[0]) - 1, 0, proc.nx - 1))
Yc = 1e3 * model.Y[ix0, :, :]
Zcs = 1e3 * model.Z[ix0, :, :]
im1 = axs[1].pcolormesh(Yc, Zcs, T_fr[0][ix0, :, :], shading='auto', cmap='inferno', vmin=T_vmin, vmax=T_vmax)
ct1 = axs[1].contour(Yc, Zcs, T_fr[0][ix0, :, :], levels=[mat.T_solidus, mat.T_liquidus], colors=['c', 'lime'])
fig.colorbar(im1, ax=axs[1], label='T [K]')
axs[1].set_xlabel('y [mm]'); axs[1].set_ylabel('z [mm]'); axs[1].set_aspect('equal')
title1 = axs[1].set_title(r'T cross-section $x\approx x_c$')

im2 = axs[2].pcolormesh(Xt, Yt, T_fr[0][:, :, -1], shading='auto', cmap='inferno', vmin=T_vmin, vmax=T_vmax)
ct2 = axs[2].contour(Xt, Yt, T_fr[0][:, :, -1], levels=[mat.T_solidus, mat.T_liquidus], colors=['c', 'lime'])
fig.colorbar(im2, ax=axs[2], label='T [K]')
axs[2].set_xlabel('x [mm]'); axs[2].set_ylabel('y [mm]'); axs[2].set_aspect('equal')
title2 = axs[2].set_title('T top surface')


def update(i):
    Ti = T_fr[i]
    im0.set_array(Ti[:, j0, :].ravel())
    arc_line.set_xdata([1e3 * arc_fr[i], 1e3 * arc_fr[i]])
    title0.set_text(f'T mid-plane, t={times[i]:.2f}s')

    ix = int(np.clip(np.searchsorted(model.xc, arc_fr[i]) - 1, 0, proc.nx - 1))
    im1.set_array(Ti[ix, :, :].ravel())
    title1.set_text(f'T cross-section, xc={1e3*arc_fr[i]:.1f} mm')

    im2.set_array(Ti[:, :, -1].ravel())
    title2.set_text(f'T top, t={times[i]:.2f}s')

    # redraw solidus/liquidus contours
    for ax in axs:
        for coll in list(ax.collections):
            if coll not in (im0, im1, im2):
                try:
                    coll.remove()
                except ValueError:
                    pass
    axs[0].contour(Xc, Zc, Ti[:, j0, :], levels=[mat.T_solidus, mat.T_liquidus], colors=['c', 'lime'])
    Yci = 1e3 * model.Y[ix, :, :]
    Zcsi = 1e3 * model.Z[ix, :, :]
    axs[1].contour(Yci, Zcsi, Ti[ix, :, :], levels=[mat.T_solidus, mat.T_liquidus], colors=['c', 'lime'])
    axs[2].contour(Xt, Yt, Ti[:, :, -1], levels=[mat.T_solidus, mat.T_liquidus], colors=['c', 'lime'])
    return (im0, im1, im2, arc_line, title0, title1, title2)


anim_T = FuncAnimation(fig, update, frames=len(T_fr), interval=150, blit=False)
plt.close(fig)
HTML(anim_T.to_jshtml())


In [ ]:
# f_L mid-plane + |V| mid-plane + pool size curves with time cursor
fig2, axs2 = plt.subplots(1, 3, figsize=(12, 3.4), constrained_layout=True)

im_f = axs2[0].pcolormesh(Xc, Zc, fL_fr[0][:, j0, :], shading='auto', cmap='Blues', vmin=0, vmax=1)
fig2.colorbar(im_f, ax=axs2[0], label=r'$f_L$')
axs2[0].set_xlabel('x [mm]'); axs2[0].set_ylabel('z [mm]'); axs2[0].set_aspect('equal')
t_f = axs2[0].set_title(rf'$f_L$ mid-plane, t={times[0]:.2f}s')

im_s = axs2[1].pcolormesh(Xc, Zc, spd_fr[0][:, j0, :], shading='auto', cmap='viridis', vmin=0, vmax=spd_vmax)
fig2.colorbar(im_s, ax=axs2[1], label='|V| [m/s]')
axs2[1].set_xlabel('x [mm]'); axs2[1].set_ylabel('z [mm]'); axs2[1].set_aspect('equal')
t_s = axs2[1].set_title(f'|V| mid-plane, t={times[0]:.2f}s')

axs2[2].plot(times, 1e3 * np.array(diag.pool_depth), label='depth')
axs2[2].plot(times, 1e3 * np.array(diag.pool_length), label='length')
axs2[2].plot(times, 1e3 * np.array(diag.pool_width), label='width')
cursor = axs2[2].axvline(times[0], color='k', ls='--', lw=1)
axs2[2].set_xlabel('t [s]'); axs2[2].set_ylabel('mm')
axs2[2].legend(fontsize=8); axs2[2].grid(True, alpha=0.3)
axs2[2].set_title('Pool size')


def update2(i):
    im_f.set_array(fL_fr[i][:, j0, :].ravel())
    im_s.set_array(spd_fr[i][:, j0, :].ravel())
    t_f.set_text(rf'$f_L$ mid-plane, t={times[i]:.2f}s')
    t_s.set_text(f'|V| mid-plane, t={times[i]:.2f}s')
    cursor.set_xdata([times[i], times[i]])
    return (im_f, im_s, cursor, t_f, t_s)


anim_flow = FuncAnimation(fig2, update2, frames=len(fL_fr), interval=150, blit=False)
plt.close(fig2)
HTML(anim_flow.to_jshtml())


## Checks

- Heat balance `|Q_in − Q_loss − ΔH| / Q_in` reported above
- \(v=0\) on the symmetry plane
- Divergence after projection is machine-zero in the liquid
- For \(d\sigma/dT < 0\) surface flow leaves the hot spot


## Checks

- Heat balance `|Q_in − Q_loss − ΔH| / Q_in` reported above
- \(v=0\) on the symmetry plane
- Divergence after projection is machine-zero in the liquid
- For \(d\sigma/dT < 0\) surface flow leaves the hot spot
